<a href="https://colab.research.google.com/github/Iberasa3/BlackAndOchre/blob/main/Avispas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [63]:
import pandas as pd

# El separador '\t' le dice a pandas que use el tabulador
df = pd.read_csv('avispas_avistamientos.csv', sep='\t') #Duh el separador era esta shit
print(f"Número de columnas: {df.shape[1]}")
print(f"Nombres de las columnas: {df.columns.tolist()[:5]}...")

Número de columnas: 50
Nombres de las columnas: ['gbifID', 'datasetKey', 'occurrenceID', 'kingdom', 'phylum']...


Efectivamente este es mi proyecto de avispas, de momento no tengo mucho más que añadir. Cuando termine de limpiar este dataset proyectaré las coordenadas en GEE

Un par de apuntes para mí mismo, probablemente acabemos considerando a todas las subespecies de avispa asiatica como una sola para no tener que liarnos entre las distintas subespecies.

Varios factores que tengo que tener en cuenta para poder filtrar bien los datos que se me están presentando:

1- Los datos anteriores a 2005, si los hay, no hay que tenerlos en cuenta porque Vespa velutina entró a europa en 2004 (¿hecho?)

2- Cuidado con null island (hecho)

3- cuidado con los rangos en los que el avistamiento tiene gran índice de error (hecho)

4- cuidado con los avistamientos no-salvajes (hecho, preservados eliminados)

5- hay que chekiar las diferentes subespecies por si acaso (hecho)

6- hay que chekiar solo los que sean de europa (hecho)


Tengo que leer algunos papers para que esto parezca más académico, pero tampoco pasarme 150 días leyendo papers para justificar toda la literatura


In [64]:
df['infraspecificEpithet'].unique() #Si la celda está vacía entonces asumimos que es la versión común de la vespa

array([nan, 'nigrithorax', 'variana', 'auraria', 'flavitarsus',
       'velutina', 'karnyi', 'celebensis', 'ardens', 'sumbana',
       'divergens', 'floresiana', 'timorensis'], dtype=object)

In [65]:
print(df['infraspecificEpithet'].value_counts(dropna=False))

infraspecificEpithet
NaN            306920
nigrithorax     70676
auraria           101
variana            46
flavitarsus        42
velutina           25
ardens             24
timorensis         16
karnyi             15
floresiana         13
celebensis         10
divergens           6
sumbana             5
Name: count, dtype: int64


In [66]:
paises_europa = ['ES', 'FR', 'PT', 'BE', 'IT', 'DE', 'CH', 'LU', 'GB', 'IE', 'NL'] #El análisis se basa en europa así que nos quedamos solo con los países europeos. La mayoría de registros de todas formas son europeos.
df = df[df['countryCode'].isin(paises_europa)]
print(f"Pasamos de {len(df)} a {len(df)} registros.")

Pasamos de 372895 a 372895 registros.


In [67]:
# Contamos cuántos registros hay por país y lo ordenamos de mayor a menor
conteo_paises = df['countryCode'].value_counts()

print("--- Avistamientos por País en Europa ---")
print(conteo_paises)

--- Avistamientos por País en Europa ---
countryCode
BE    201699
NL     72038
FR     54466
CH     20092
DE     17921
ES      3518
PT      2251
LU       642
IT       211
IE        37
GB        20
Name: count, dtype: int64


In [68]:
#Quitamos todas las coordenadas basura y los nulls
df = df[(df['decimalLatitude'] != 0.0) & (df['decimalLongitude'] != 0.0)]
df = df.dropna(subset=['decimalLatitude', 'decimalLongitude'])

print(f"Registros originales: {len(df)}")
print(f"Registros después de haber quitado los nulls y los null-islands: {len(df)}")

Registros originales: 372450
Registros después de haber quitado los nulls y los null-islands: 372450


In [69]:
#Quitamos las instancias en las que la incertidumbre es superior a 1km. Habría que ver por qué coño existen incertidumbres tan grandes
umbral_maximo = 1000
df = df[(df['coordinateUncertaintyInMeters'] <= 1000) & (df['year'] >= 2005)]

Hay que tener cuidado con las instancias que se hayan tomado en el mismo instante, puede ser que un pibe haya apuntado varias veces a una misma colmena o incluso a una misma avispa, y que por ello haya zonas sobrerepresentadas.
Ahora habría que ver cómo chota hacemos esto del spatial thinning.
Como hemos descargado la versión simple del Darwin Core igual tenemos datos que nos ayudan con esto. De hecho igual en un futuro lo hago directamente con DWC

In [70]:
df = df[df['basisOfRecord'] != 'PRESERVED_SPECIMEN']
print(df['basisOfRecord'].unique())

['HUMAN_OBSERVATION']


Vamos con el spatial thinning.

Podemos aplicar una agrupación por celda, útil para evitar sobremuestreos pero falla para detectar zonas más idóneas en las que las avispas hayan podido construir sus colmenas. Luego podemos sumarlo a un check de correlación para ver si nos ha salido bien.

La cosa es que si vamos a aplicar un random forest se asume que cada dato va a ser independiente, el modelo no va a saber que los distintos puntos que están cerca entre sí pueden reflejar una colmena próxima. Además al reflejar los puntos en el propio mapa la resolución no va a dar para más, así que creo que hacer el spatial thinning sin tener en cuenta la posibilidad de colmena es, de momento, suficiente. Esto es un modelo de distribución, no uno de abundancia.

¿Cuánto vuela una avispa desde su nido?
Por lo que he leído en internet de fuentes fiables, todas coinciden en que rara vez sobrepasan 1km o el kilometro y medio desde sus nidos, así que si hay algún avistamiento de avispa debe de haber un nido relativamente cerca casi seguro



In [71]:
num_duplicados = df.duplicated().sum()
print(f"Registros con el mismo gbifID: {num_duplicados}")
df.drop_duplicates(subset='gbifID', inplace=True)

Registros con el mismo gbifID: 0


Hemos comprobado también que no haya avistamientos con un ID repetido. Se asume que los estándares de Darwin Core no permiten duplicados en sus datasets, pero por si acaso

In [72]:
# 0.015 grados de latitud son aproximadamente 1.1 km, así que cada celda va a tener aprox 1,1km x 1,1km
# Es una resolución estándar para modelos climáticos de 1km.
res = 0.015

# Dividimos por la resolución, redondeamos al entero más cercano y volvemos a multiplicar.
df['lat_grid'] = (df['decimalLatitude'] / res).round() * res
df['lon_grid'] = (df['decimalLongitude'] / res).round() * res

print(f"originalmente hay {len(df)} registros")
df = df.drop_duplicates(subset=['lat_grid', 'lon_grid'], keep='first').copy()

print(f"Dataset tras thinning (1km): {len(df)} registros")

df[['decimalLatitude', 'lat_grid', 'decimalLongitude', 'lon_grid']].head()

originalmente hay 231961 registros
Dataset tras thinning (1km): 29307 registros


,decimalLatitude,lat_grid,decimalLongitude,lon_grid
2,51.11106,51.105,4.07784,4.080
3,51.21826,51.225,2.90049,2.895
4,50.86031,50.865,4.62720,4.620
5,51.03947,51.045,3.74158,3.735
6,50.95183,50.955,3.68754,3.690


Nos hemos quitado casi el 90% del dataset eliminando avistamientos extremadamente cercanos. Ahora tenemos 29.300 registros para entrenar nuestro modelo

Eso sí, lo mejor tratar de forma distinta los datos que tengan poca fiabilidad y los datos que sean muy exactos en su radio de avistamiento?... Lo que ocurre es que las avispas vuelan, y pueden volar ocasionalmente a zonas que no le son adecuadas para nidificar.


In [ ]:
# Contamos cuántos registros hay por país y lo ordenamos de mayor a menor, estos son las casillas donde ha habido avistamiento
conteo_paises = df['countryCode'].value_counts()
conteo_total = df
print("--- Avistamientos por País en Europa ---")
print(conteo_paises)

In [73]:
import sys
import geemap
import ee # Import the Earth Engine library

# Install geojson library if not already installed
!pip install geojson

# Authenticate and Initialize Earth Engine
ee.Authenticate()
ee.Initialize(project='vespa-489513')

columnas_necesarias = ['decimalLatitude', 'decimalLongitude', 'year']
df = df[columnas_necesarias].copy()

df = df.dropna(subset=['decimalLatitude', 'decimalLongitude']) #Eliminamos los valores na

puntos_geospatiales = geemap.pandas_to_ee( #Estos sin mis puntos según la latitud y longitud a partir de ahora
    df,
    latitude='decimalLatitude',
    longitude='decimalLongitude'
)

"""
Map = geemap.Map()
Map.addLayer(puntos_geospatiales, {'color': 'red'}, 'Vespa velutina (Optimized)')
Map.centerObject(puntos_geospatiales, 6)
Map
"""

"\nMap = geemap.Map()\nMap.addLayer(puntos_geospatiales, {'color': 'red'}, 'Vespa velutina (Optimized)')\nMap.centerObject(puntos_geospatiales, 6)\nMap\n"

Ahora toca hacer que un modelo randomForest me aprenda las características de los sitios donde puede haber más avispas Vespa Velutina.

Hasta aquí hemos:
1- Conseguido el dataset desde GBIF, que registra observaciones de la Vespa velutina en un formato Darwin core (aunque de momento con la versión procesada nos sirve)

2- Hemos filtrado el dataset (1) quedándonos solo con los países de europa (las condiciones de Asía no nos sirven si son parecidas?¿?), (2) filtrando y eliminando las coordenadas no-útiles, (3) eliminando las instancias cuya tasa de error es superior a 1km, (4) quedándonos solo con las especies vistas en estado silvestre, (5) quedándonos solo con los avistamientos realizados a partir de 2005, y (6), eliminando las posibles observaciones duplicadas.

3- Hemos hecho una refracción espacial de 1,6km para que ciertos avistamientos no se vean representados, convirtiendo el mapa en casillas cuyos valores son o 1 (en cuyo caso la avispa está presente) o 0, en cuyo caso no lo está

4- Hemos reflejado dichas casillas en el mapa con GEE.

**El siguiente paso es aprender los valores geográficos de las casillas en las que SÍ hay avispa y de las casillas en las que NO las hay. Luego aplicaremos un modelo random forest para poder predecir futuras expansiones. La casilla 0 estará representada por pseudoausencias, que enseñarán al modelo en qué condiciones NO hay avispas.**

PROBLEMA: ¿CÓMO SABEMOS SI NO HAY AVISPAS PORQUE TODAVÍA NO HAN LLEGADO O PORQUE LAS CONDICIONES NO SON FAVORABLES? Se soluciona con SM4

Usaremos WorldClim para las variables bioclimáticas, NASA SRTM para valores de presión y altura, y MODIS/WorldCover para saber el tipo de cobertura terrestre sobre la que estamos trabajando.

¿Es esto útil u óptimo tras haber simplifcado cada punto a una casilla? en 1km puede cambiar mucho la altura y la distribución de suelo...
y esto de arriba no tiene mucho sentido porque con el thinning nos quedamos siempre con un punto.

--------------------------------------------

Vale, vamos con el feature engineering:
Hay una serie de variables que nos puede interesar contemplar, NO para meterlos en el modelo tal cual, pero si para ver si son lo suficientemente relevantes como para poder meterlos después. Como la literatura es más o menos clara en las variables que nos interesan podemos prescindir de hacer un PCA y hacer en lugar de eso un análisis de multicolinealidad más sencillito con VIF.

WORLDCLIM **bio01** (temperatura media), **bio05** (máxima temepratura del mes más calido), **bio06** (minima temperatura del mes más frio), **bio07** (rango de temperaturas anual), **bio12** (precipitaciones anuales), **bio13** (precipitacion del mes más humedo), **bio14** (precipitacion del mes menos humedo), **bio 15** (rango de humedad anual), **NASA SRTM elevation** (altura), **NASA SRTM slope** (cuesta), ESA WorldCover (tipo de terreno), OpenStreetMap / GEE (distancia a carreteras o vias urbanas)

In [74]:
#CONFIGURACIÓN DE PREDICTORES

aoi = puntos_geospatiales.geometry().bounds().buffer(50000)

#Clima
bioclim = ee.Image('WORLDCLIM/V1/BIO').clip(aoi)

#Terreno
srtm = ee.Image('USGS/SRTMGL1_003').clip(aoi)
elevacion = srtm.select('elevation')
slope = ee.Terrain.slope(srtm).rename('slope')

#Tipo de suelo
terreno = ee.ImageCollection("ESA/WorldCover/v100").first().clip(aoi).select(['Map'], ['landcover'])

#Distancia a carreteras u otras zonas urbanas.
dist_roads = terreno.eq(50).fastDistanceTransform().sqrt().multiply(1000).rename('dist_roads')


# Fusionamos todas las bandas en una sola imagen multidimensional
predictores = bioclim.addBands(elevacion).addBands(slope).addBands(terreno).addBands(dist_roads).select([
    'bio01', 'bio05', 'bio06', 'bio07',  'bio12', 'bio13', 'bio14', 'bio15', 'elevation', 'slope', 'landcover', 'dist_roads'
])

# SAMPLING
# Cada punto ahora tiene asociados un  clima + Relieve + Suelo + Proximidad vial
datos_presencia_mureados = predictores.sampleRegions(
    collection=puntos_geospatiales,
    properties=['year'],
    scale=1000,
    geometries=True
)

Esta sección aquí comentada es para previsualizar las distintas capas que estamos captando. Descomentarla si se quiere chekiar.

In [ ]:

"""
import geemap
import ee

predictores = bioclim.addBands(elevacion).addBands(slope).addBands(terreno).addBands(dist_roads).select([
    'bio01', 'bio05', 'bio06', 'bio07', 'bio12', 'bio13', 'bio14', 'bio15', 'elevation', 'slope', 'landcover', 'dist_roads'
])


# Paleta para Temperaturas (Bio01, Bio06, etc.) - De azul frío a rojo calor
vis_temp = {'min': -50, 'max': 300, 'palette': ['blue', 'cyan', 'green', 'yellow', 'red']}

# Paleta para Precipitaciones (Bio12, Bio14) - De blanco seco a azul lluvia
vis_precip = {'min': 0, 'max': 2500, 'palette': ['white', 'lightblue', 'blue', 'darkblue']}

# Paleta para Elevación
vis_el = {'min': 0, 'max': 2500, 'palette': ['006600', '002200', 'fff700', 'ab7634', 'c7d6ec', 'ffffff']}

# Paleta para Pendiente (Slope) - De verde suave a rojo escarpado
vis_slope = {'min': 0, 'max': 45, 'palette': ['white', 'orange', 'red']}

# Paleta oficial ESA WorldCover (Landcover)
vis_lc = {'bands': ['landcover'], 'min': 10, 'max': 100,
          'palette': ['006400', 'ffbb22', 'ffff4c', 'f096ff', 'fa0000', 'b4b4b4', 'f0f0f0', '0064ff', '00bb88', 'ffff4c']}

# Paleta para Distancia a Carreteras - De rojo (cerca) a blanco (lejos)
vis_dist = {'min': 0, 'max': 10000, 'palette': ['red', 'orange', 'white']}

Map = geemap.Map()
Map.add_basemap('HYBRID')

# Añadimos las capas una a una (puedes activarlas/desactivarlas en el menú de capas)
Map.addLayer(predictores.select('bio01'), vis_temp, 'Temperatura Media Anual (Bio01)', False)
Map.addLayer(predictores.select('bio06'), vis_temp, 'Temp. Mínima Invierno (Bio06)', False)
Map.addLayer(predictores.select('bio12'), vis_precip, 'Precipitación Anual (Bio12)', False)
Map.addLayer(predictores.select('elevation'), vis_el, 'Elevación (SRTM)', False)
Map.addLayer(predictores.select('slope'), vis_slope, 'Pendiente (Slope)')
Map.addLayer(predictores.select('landcover'), vis_lc, 'Tipo de Suelo (ESA WorldCover)')
Map.addLayer(predictores.select('dist_roads'), vis_dist, 'Proximidad a Zonas Urbanas/Viales')

# Añadimos tus puntos de presencia como referencia visual
Map.addLayer(puntos_geospatiales, {'color': 'magenta'}, 'Nidos Reales (Puntos)')

# Centrar el mapa
Map.centerObject(puntos_geospatiales, 7)
Map
"""


En esta sección llamamos a mi script VIF_variable_selector para que haga una selección de las variables que mejor representan los datos entre todas las variables que hemos considerado posiblemente útiles.

In [ ]:
import geemap
import pandas as pd

from VIF_variable_selector import run_vif_cleaner

print("Bajando datos de GEE...")
df_total = geemap.ee_to_df(datos_presencia_mureados)

# Definimos el umbral a 10, aunque podríamos meterlo a 5 también.
threshold = 10.0

print("Iniciando purga de variables redundantes...")
variables_finales, reporte_vif = run_vif_cleaner(df_total, threshold)

print("\n--- SUBSET DE ORO ---")
print(f"Variables supervivientes: {variables_finales}")
print("\nValores VIF finales:")
print(reporte_vif)

Esto de abajo es para visualización de los valores VIF

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

data = {
    'Variable': ['bio05', 'bio06', 'bio13', 'bio14', 'elevation', 'bio01', 'bio15', 'slope', 'bio07', 'bio12', 'dist_roads', 'landcover'],
    'VIF': [500, 104.91, 24.71, 10.16, 2.86, 2.36, 2.34, 1.80, 1.61, 1.35, 1.31, 1.11],
    'Subset': ['Eliminada', 'Eliminada', 'Eliminada', 'Eliminada', 'Oro', 'Oro', 'Oro', 'Oro', 'Oro', 'Oro', 'Oro', 'Oro']
}

df = pd.DataFrame(data)
# Invertimos el orden para que las "Oro" salgan arriba en el gráfico horizontal
df = df.iloc[::-1]

colors = ['#e74c3c' if s == 'Eliminada' else '#3498db' for s in df['Subset']]

plt.figure(figsize=(10, 8)) # Ajustamos el tamaño para que sea más alto que ancho

# Cambiamos bar por barh (h de horizontal)
bars = plt.barh(df['Variable'], df['VIF'], color=colors, edgecolor='black', alpha=0.8)

# Ahora la escala logarítmica va en el eje X
plt.xscale('log')

# Ajustamos la posición del texto para el formato horizontal
for bar in bars:
    xval = bar.get_width()
    plt.text(xval * 1.1, bar.get_y() + bar.get_height()/2,
             f'{xval if xval < 400 else "inf"}',
             va='center', ha='left', fontsize=10, fontweight='bold')

# Cambiamos la línea vertical (vlines) en lugar de horizontal (hlines)
plt.axvline(x=10, color='gray', linestyle='--', alpha=0.6, label='Threshold (VIF=10)')

plt.title('VIF Analysis: Feature Selection for Vespa velutina Model', fontsize=14, pad=20)
plt.xlabel('VIF Value (Log Scale)')
plt.ylabel('Environmental Predictors')

plt.grid(axis='x', linestyle=':', alpha=0.5)

# Leyenda personalizada
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], color='#3498db', lw=4, label='Golden Subset (Accepted)'),
                   Line2D([0], [0], color='#e74c3c', lw=4, label='Redundant (Dropped)')]
plt.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()

In [84]:
# Estos son los que mantuvieron un VIF < 10 (ESTA ES MI COLECCIÓN FINAL DE PRESENCIAS)
variables_oro = [
    'elevation', 'bio01', 'bio15', 'slope',
    'bio07', 'bio12','dist_roads', 'landcover'
] #dist_roads causa bugs, luego lo volvemos a meter si eso

predictores_final = predictores.select(variables_oro)
presencias_final = predictores_final.sampleRegions(
    collection=puntos_geospatiales,
    properties=['year'],
    scale=1000,
    geometries=True
)

presencias_final = presencias_final.map(lambda f: f.set('class', 1))

----------------------------------------------------------------

Esta sección es para enseñarle al modelo dónde NO hay avispas, para ello generamos pseudoausencias. En un principio tenía pensado hacerlas random, pero rápidamente surjen varios problemas:

1- qué pasa si en la zona de avistamiento hay un punto de ausencia cuando en realidad debería de ser de presencia? A lo mejor ese lugar es un bosque profundo y no pasan personas para avistar avispas. O igual el punto random cae en un sitio en el que sí ha habido avistamientos. Hacerlo random, sobre todo teniendo en cuenta que lo hacemos en un rango aoi donde sí se han producido avistamientos, no es una buena idea

2- Si efectivamente el punto random cae en un sitio donde efectivamente la avispa está ausente, ¿cómo sabemos que está ausente porque las condiciones no son las adecuadas? a lo mejor simplemente la avispa todavía no ha llegado a esa zona, o hay una barrera geográfica que le impide llegar aunque las condiciones sí sean las adecuadas en la zona en sí (imaginemos el paraíso de las avispas rodeado de unas montañas inexpugnables, la avispa no llegará al paraíso pero no porque el paraiso no sea bueno, si no porque no tiene forma de llegar ahí).

Haremos un híbrido y luego lo justificaremos en el documento y en el vídeo.

## SM1

Estrategia para generar pseudoausencias 1: SM1 random
Generamos puntos random en nuestra zona de interés que no coincidan con los puntos de presencia. Una vez generados los visualizamos y guardamos en un dataset para poder compararlos en un futuro y no tener que ejecutar esta sección cada vez.

In [85]:

#GENERACIÓN DE AUSENCIAS SM1 (IMPORTANTE CHEKIARLO PARA VER MI COLECCIÓN DE PSEUDOAUSENCIAS)

aoi = puntos_geospatiales.geometry().bounds().buffer(2000000) #aoi devuelve una figura geométrica de 2000km alrededor de los puntos

#Creamos la imagen de los puntos de presencia
presencia_img = ee.Image().paint(puntos_geospatiales, 1000)

# Creamos la máscara para evitar el oceano
landcover = ee.ImageCollection("ESA/WorldCover/v100").first()
mascara_tierra = landcover.neq(80)

# Creamos una imagen de valor 1 en toda la AOI y "borramos" donde hay presencias
# Usamos un buffer de 1000m (1km) para asegurar que no caigan en el mismo píxel por si acaso

mascara_exclusion = presencia_img.fastDistanceTransform().sqrt().gt(1)


area_muestreo = ee.Image(1).clip(aoi).updateMask(mascara_exclusion).updateMask(mascara_tierra) #Area muestreo es el área permitida donde puede haber ausencias

# Generamos las 29.306 pseudo-ausencias (Seguimos un ratio 1:1), aunque se pueden generar algunos menos
# si el punto random cae justo en una presencia y se descarta.

pseudo_ausencias_raw = predictores_final.updateMask(area_muestreo).sample(
    region=aoi,
    scale=1000,
    numPixels=1000000, #Esto se trunca con el raw_limit
    seed=67,
    geometries=True
)

# Etiquetamos y unimos el dataset
# Clase 1 = Presencia, Clase 0 = Pseudo-ausencia
# 2. Etiquetamos las ausencias como Clase 0
# Importante: Estas ya traen las variables de 'predictores_final' por el paso anterior
ausencias = pseudo_ausencias_raw.limit(28887).map(lambda f: f.set('class', 0))

# 3. Usamos las presencias que YA muestreamos antes (las que tienen variables_oro)
# Asegúrate de que este objeto existe en tu código (lo creaste en la celda anterior)
presencias_final = presencias_final.map(lambda f: f.set('class', 1))

# 4. Unimos los dos datasets "completos"
dataset_total_SM1 = presencias_final.merge(ausencias)

"""
print(f"Dataset SM1 listo. AUSENCIAS: {ausencias.size().getInfo()}, deberían de ser cerca de 29306)")
print(f"Dataset SM1 listo. PRESENCIAS:: {presencias_final.size().getInfo()}, deberían de ser cerca de 29306)")
#print(f"Dataset SM1 listo. Total puntos: {dataset_total_SM1.size().getInfo()}, deberían de ser cerca de 58.512")
"""

'\nprint(f"Dataset SM1 listo. AUSENCIAS: {ausencias.size().getInfo()}, deberían de ser cerca de 29306)")\nprint(f"Dataset SM1 listo. PRESENCIAS:: {presencias_final.size().getInfo()}, deberían de ser cerca de 29306)")\n#print(f"Dataset SM1 listo. Total puntos: {dataset_total_SM1.size().getInfo()}, deberían de ser cerca de 58.512")\n'

In [ ]:

# SECCIÓN DE VISUALIZACIÓN SM1
Map = geemap.Map()
Map.add_basemap('HYBRID')

estilo_vespa = {'color': 'red', 'pointSize': 4, 'pointShape': 'circle', 'width': 1}
estilo_vacio = {'color': '00ffff', 'pointSize': 4, 'pointShape': 'square', 'width': 1} # Cian más grande

# Metemos una opción de máscara para que se vea en qué región se pueden colocar los puntos
vis_mask = {
    'min': 0,
    'max': 1,
    'palette': ['000000', 'yellow'], # Changed 'transparent' to '000000' (black)
    'opacity': 0.35 # Menos opaco para ver el terreno
}
Map.addLayer(area_muestreo, vis_mask, 'Debug: Área de Muestreo (Permitida)')
Map.addLayer(ausencias.style(**estilo_vacio), {}, 'Pseudo-ausencias (Clase 0)')
Map.addLayer(presencias.style(**estilo_vespa), {}, 'Presencias Vespa (Clase 1)')

Map.centerObject(puntos_geospatiales, 6)
Map


In [88]:
import geemap

# SCRIPT DE DEBUG Y VISUALIZACIÓN PARA PUNTOS SM1 CON DATOS

# 1. Separar para verificar (esto es lo que pedías)
puntos_presencia = dataset_total_SM1.filter(ee.Filter.eq('class', 1))
puntos_ausencia = dataset_total_SM1.filter(ee.Filter.eq('class', 0))

# 2. Imprimir diagnóstico en la consola
print(f"✅ Presencias finales: {puntos_presencia.size().getInfo()}")
print(f"✅ Ausencias finales: {puntos_ausencia.size().getInfo()}")

# 3. COMPROBACIÓN CRÍTICA DE COLUMNAS
# Esto nos dirá si ambos grupos tienen las mismas variables_oro
print("Columnas en Presencia:", presencias_final.first().propertyNames().getInfo())
print("Columnas en Ausencia:", puntos_ausencia.first().propertyNames().getInfo())

# 4. Visualización en el Mapa
Map = geemap.Map()

# --- ESTA ES LA LÍNEA QUE EVITA EL ERROR 403 ---
Map.add_basemap('HYBRID') # Usa satélite de Google en lugar de OpenStreetMap
# -----------------------------------------------

# Dibujamos las ausencias en ROJO (pequeñas para no tapar todo)
Map.addLayer(puntos_ausencia,
             {'color': 'red', 'pointSize': 1},
             'Pseudo-ausencias (Clase 0)')

# Dibujamos las presencias en AZUL (más grandes para que resalten)
Map.addLayer(puntos_presencia,
             {'color': 'blue', 'pointSize': 3},
             'Presencias Reales (Clase 1)')

Map.centerObject(puntos_geospatiales, 5)
Map

✅ Presencias finales: 28887
✅ Ausencias finales: 28887
Columnas en Presencia: ['class', 'system:index', 'year', 'elevation', 'landcover', 'dist_roads', 'bio07', 'bio15', 'slope', 'bio01', 'bio12']
Columnas en Ausencia: ['system:index', 'class', 'elevation', 'bio01', 'bio15', 'slope', 'bio07', 'bio12', 'dist_roads', 'landcover']


Map(center=[48.71174353299487, 3.7533902755953057], controls=(WidgetControl(options=['position', 'transparent_…

Exportamos las pseudoausencias generadas con SM1 para no tener que hacer el proceso de limpieza cada vez y luego poder comparar los modelos utilizando diferentes técnicas.
Esto lo mantengo comentado por si acaso tengo que volver a hacerlo.



In [ ]:

# EXTRACCIÓN Y GUARDADO DE COORDENADAS (APLICABLE PARA TODAS CAMBIANDO UN PAR DE COSILLAS)

def extraer_posicion(feature):
    coords = feature.geometry().coordinates()
    return feature.set({
        'latitude': coords.get(1),
        'longitude': coords.get(0)
    })


ausencias_finales = ausencias.map(extraer_posicion).select(['latitude', 'longitude', 'class'])
nombre_csv = 'SM1_absences_COORDENADASDef.csv'
geemap.ee_to_csv(ausencias_finales, filename=nombre_csv)


## SM2

Estrategia para generar pseudoausencias 2: SM2 límite geográfico
Generamos puntos random en nuestra zona de interés que no coincidan con los puntos de presencia, pero SOLO en un área suficientemente cercana a nuestros puntos para obligar al modelo a aprender de zonas que estén relativamente cerca de nuestra área. No tiene tanto impacto porque se solapa bastante con aoi. Una vez generados los visualizamos y guardamos en un dataset para poder compararlos en un futuro y no tener que ejecutar esta sección cada vez.

In [ ]:
# GENERACIÓN DE AUSENCIAS SM2 (SPATIALLY CONSTRAINED)

# Definimos el AOI como una "nube" de buffers de 350km alrededor de los nidos
# Quitamos.bounds() para que no sea un rectángulo, sino el área real de influencia

aoi_sm2 = puntos_geospatiales.geometry().buffer(350000)

presencia_img = ee.Image().paint(puntos_geospatiales, 1)

landcover = ee.ImageCollection("ESA/WorldCover/v100").first()
mascara_tierra = landcover.neq(80)

mascara_exclusion = presencia_img.fastDistanceTransform().sqrt().gt(1)

area_muestreo_sm2 = ee.Image(1).clip(aoi_sm2).updateMask(mascara_exclusion).updateMask(mascara_tierra)

pseudo_ausencias_sm2_raw = area_muestreo_sm2.sample(
    region=aoi_sm2,
    scale=1000,
    numPixels=60000, #Esto se trunca a 29306
    seed=67,
    geometries=True
)

pseudo_ausencias_sm2 = pseudo_ausencias_sm2_raw.limit(29306)

# Etiquetamos y unimos el dataset
presencias = puntos_geospatiales.map(lambda f: f.set('class', 1))
ausencias_sm2 = pseudo_ausencias_sm2.map(lambda f: f.set('class', 0))

# El dataset final para el modelo SM2
dataset_total_sm2 = presencias.merge(ausencias_sm2)

print(f"Dataset SM2 listo con: {dataset_total_sm2.size().getInfo()} puntos.")
print("La IA ahora solo aprenderá de ausencias situadas a menos de 350km de los nidos.")

In [ ]:
# SECCIÓN DE VISUALIZACIÓN SM2

Map = geemap.Map()
Map.add_basemap('HYBRID')

# Estilos definidos para Vespa y Ausencias
estilo_vespa = {'color': 'red', 'pointSize': 4, 'pointShape': 'circle', 'width': 1}
estilo_vacio = {'color': '00ffff', 'pointSize': 4, 'pointShape': 'square', 'width': 1}

# Configuración de la máscara de búsqueda (Nube de 350km)
# Usamos unmask(0) para que los "huecos" de 1km se vean negros
# El clip(aoi_sm2) asegura que solo veamos la forma de la nube y no un rectángulo
vis_mask = {
    'min': 0,
    'max': 1,
    'palette': ['000000', 'yellow'], # Negro para prohibido, Amarillo para permitido
    'opacity': 0.45 # Ajustado para que sea vistoso pero deje ver el mapa
}

# Representamos la zona de muestreo permitida para SM2
Map.addLayer(area_muestreo_sm2.unmask(0).clip(aoi_sm2), vis_mask, 'Región de Muestreo SM2 (350km)')

# Añadir los puntos (rasterizados para fluidez)
Map.addLayer(ausencias_sm2.style(**estilo_vacio), {}, 'Pseudo-ausencias SM2 (Clase 0)')
Map.addLayer(presencias.style(**estilo_vespa), {}, 'Presencias Vespa (Clase 1)')

# Centrar el mapa en la invasión
Map.centerObject(puntos_geospatiales, 6)
Map

In [ ]:
"""
# EXTRACCIÓN Y GUARDADO DE COORDENADAS (APLICABLE PARA TODAS CAMBIANDO UN PAR DE COSILLAS)

def extraer_posicion(feature):
    coords = feature.geometry().coordinates()
    return feature.set({
        'latitude': coords.get(1),
        'longitude': coords.get(0)
    })


ausencias_finales = ausencias_sm2.map(extraer_posicion).select(['latitude', 'longitude', 'class'])
nombre_csv = 'SM2_absences_COORDENADASDef.csv'
geemap.ee_to_csv(ausencias_finales, filename=nombre_csv)"""

SM3: Cambiamos de clima para forzar a que el modelo se ajuste

--------------------------------------------------------------------------
EN ESTA SECCIÓN DE AQUÍ ABAJO VAMOS A ENTRENAR EL RANDOM FOREST CON LOS MODELOS DE PRESENCIA Y CON LAS PSEUDOAUSENCIAS DEL SM1
--------------------------------------------------------------------------


In [90]:
import geemap

# Definimos una paleta de colores: de blanco/amarillo (poco probable) a rojo oscuro (muy probable)
vis_params = {
    'min': 0,
    'max': 1,
    'palette': ['#ffffff', '#fdfd96', '#ffb347', '#ff6961', '#8b0000']
}

Map = geemap.Map()
Map.add_basemap('HYBRID')

# Añadimos el mapa de fitness
# Nota: Si quieres ver probabilidades (0 a 1) en lugar de solo 0 o 1,
# asegúrate de que el clasificador esté en modo 'PROBABILITY'
prob_classifier = classifier.setOutputMode('PROBABILITY')
mapa_probabilidades = predictores_final.select(nombres_variables).classify(prob_classifier)

Map.addLayer(mapa_probabilidades, vis_params, 'Índice de Fitness (Probabilidad)')

# Añadimos tus presencias originales en azul para comparar
Map.addLayer(presencias_final, {'color': 'blue'}, 'Presencias Reales')

Map.centerObject(puntos_geospatiales, 5)
Map

Map(center=[48.71174353299487, 3.7533902755953057], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
# Extraemos la importancia de las variables
importancia = classifier.explain().get('importance').getInfo()

# Lo convertimos a un formato más legible (ordenado de mayor a menor)
importancia_ordenada = dict(sorted(importancia.items(), key=lambda item: item[1], reverse=True))

print("--- IMPORTANCIA DE LAS VARIABLES ---")
for var, val in importancia_ordenada.items():
    print(f"{var}: {round(val, 2)}")

# Opcional: Graficarlo rápido con matplotlib
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.bar(importancia_ordenada.keys(), importancia_ordenada.values(), color='skyblue')
plt.title('Importancia de las Variables en el Modelo (SM1)')
plt.xticks(rotation=45)
plt.show()

## REVISIONES

Cosas a arreglar o revisar:

1- el encasillamiento puede no ser óptimo, además podemos estar sobrerrepresentando zonas de clima templado atlántico porque es justo dónde más se ha preocupado la gente de registrar los avistamientos. Pero eso no quiere decir que no se puedan adaptar a otros sitios, ¿o sí?

2- las pseudoausencias, creo, deberían de ser tratadas de otra forma. SEGURO QUE DEBEN DE SER TRATADAS DE OTRA FORMA

3- podemos tratar de forma diferente los datos precisos de los no precisos? No es necesario, ya hemos filtrado los datos no precisos. Además un avistamiento puede darse a varios cientos de metros o más de una colmena.

4- hay que investigar con papers cómo se ha llevado esto a cabo por parte de otras personas. Esto lo haré a la tarde

4.2- MEJORAR EL FEATURE ENGINEERING, CALCULANDO MÁS INFO IMPORTANTE (DISTANCIA A RIOS, A CIUDADES, ILUMINACIÓN...)

5- ENTRENAR EL MODELO, LUEGO HACER VALIDACIÓN CRUZADA (SE VERÁ COMO SE HACE) PARA VER CÓMO DE BUENO ES.

6- REVISAR, DOCUMENTAR, README DE GITHUB Y PRODUCCIÓN DEL VÍDEO


Enlaces útiles:

https://ramirodcrego.com/teaching/gee/

https://developers.google.com/earth-engine/tutorials/community/species-distribution-modeling?hl=es-419

https://www.tandfonline.com/doi/full/10.1080/10095020.2025.2546507#abstract

https://www.researchgate.net/publication/284246225_A_multi-scale_approach_to_identify_invasion_drivers_and_invaders'_future_dynamics

https://onlinelibrary.wiley.com/doi/full/10.1002/ece3.70029

https://besjournals.onlinelibrary.wiley.com/doi/10.1111/j.2041-210X.2011.00172.x

https://biogeography-usc.org/positive/Prediction.html

https://www.sciencedirect.com/science/article/abs/pii/S0006320711001315

https://revistaecosistemas.net/index.php/ecosistemas/article/view/2987/1962

https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0071218 (PAPER SM4)

https://gidahatari.com/ih-es/bioclim-un-sistema-de-analisis-y-prediccion-de-bioclimas (BIOCLIM)

https://www.researchgate.net/publication/226562656_DOMAIN_a_flexible_modelling_procedure_for_mapping_potential_distributions_of_plants_and_animals (DOMAIN)

https://www.sciencedirect.com/science/article/pii/S0304380011000780 (PAPER 1 JUSTIFICACIÓN BUFFER)

https://besjournals.onlinelibrary.wiley.com/doi/10.1111/j.2041-210X.2010.00036.x (PAPER 2 JUSTIFICACIÓN BUFFER)

https://besjournals.onlinelibrary.wiley.com/doi/full/10.1111/1365-2664.12724 (PAPER 3 JUSTIFICACIÓN BUFFER)

https://www.plant-animal.es/pdfs/Herrera.2024.Ecosistemas.pdf (paper justificación presencias)